## Working with TiTiler-EoPF - Single Item (ZARR datastore)

This notebook demonstrates how to use the TiTiler-EoPF service to visualize a single Item (STAC) of EOPF Zarr datastore.


In [ ]:
# Start services
!docker compose up api -d

In [ ]:
import json
import httpx2 as httpx
from folium import Map, TileLayer

%matplotlib inline

In [ ]:
titiler_endpoint = "http://127.0.0.1:8000"

In [ ]:
r = httpx.get(f"{titiler_endpoint}/_mgmt/health")
print(r.json())

### Conformances

In [ ]:
r = httpx.get(f"{titiler_endpoint}/conformance").json()
print(json.dumps(r, indent=4))

# Find Item ID

Let's find one Item to visizualize. We can use the [EOPF STAC API](https://api.explorer.eopf.copernicus.eu/stac) to find an Item ID.

In [ ]:
collection_id = "sentinel-2-l2a"
bbox = [11.393460776835221, 41.42010137184542, 12.466572328262494, 42.426911995973605]
date = "2026-01-17T00:00:00Z/2026-08-01T23:59:59Z"

r = httpx.get(
    "https://api.explorer.eopf.copernicus.eu/stac/search",
    params={
        "collections": [collection_id],
        "bbox": ",".join(map(str, bbox)),
        "datetime": date,
    },
).json()
print("Found :", r["numberMatched"])

item = r["features"][0]
item_id = item["id"]
print("Using item: ", item_id)

## Item Info

In [ ]:
# List Availables assets for the item
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/assets",
    timeout=20,
).json()

print(json.dumps(r, indent=4))

In [ ]:
# Get Info for the `reflectance` assets
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/info",
    params={"assets": "reflectance"},
    timeout=20,
).json()

print("List of group/variable: ", list(r))
print("Info for `reflectance_b01`")
print(json.dumps(r["reflectance_b01"], indent=4))

In [ ]:
# Get Info for specific variable within the `reflectance` assets
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/info",
    params={"assets": "reflectance|bands=b01,b02"},
    timeout=20,
).json()

print("List of variable: ", list(r))
print(json.dumps(r["reflectance_b01"], indent=4))

## Get True Color Preview for the item

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/preview.png",
    params={
        "assets": "reflectance|bands=b04,b03,b02",
        "rescale": "0,0.6",
    },
    timeout=20,
)
print(r.headers)

In [ ]:
from IPython.display import Image

Image(r.content)

## Display True Color tiles

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/WebMercatorQuad/tilejson.json",
    params={
        "assets": "reflectance|bands=b04,b03,b02",
        "rescale": "0,0.6",
        "tilesize": 256,
    },
    timeout=20,
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=8
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)
m

## Display NDVI tiles

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/WebMercatorQuad/tilejson.json",
    params=[
        ("assets", "reflectance|bands=b04,b8a"),
        ("expression", "(b2-b1)/(b2+b1)"),
        ("rescale", "-1,1"),
        ("colormap_name", "viridis"),
        ("tilesize", 256),
    ],
    timeout=10,
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=9
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)

m

In [ ]:
r = httpx.get(
    f"{titiler_endpoint}/collections/{collection_id}/items/{item_id}/WebMercatorQuad/tilejson.json",
    params=[
        ("assets", "reflectance|expression=(b8a-b04)/(b8a+b04)"),
        ("rescale", "-1,1"),
        ("colormap_name", "viridis"),
        ("tilesize", 256),
    ],
    timeout=10,
).json()
print(r)
bounds = r["bounds"]
m = Map(
    location=((bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2), zoom_start=9
)

TileLayer(tiles=r["tiles"][0], opacity=1, attr="ESA EoPF").add_to(m)

m